# Churn Prediction Project.

### Data preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/workspaces/Machine-Learning-Bootcamp-by-Zoomcamp/Week3/Telecom_churn_csv data.csv')

In [ ]:
df.head()

In [ ]:
df.head().T

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ','_')
df.columns

In [ ]:
df.dtypes

In [ ]:
categorical_col = list(df.dtypes[df.dtypes =='object'].index)

In [ ]:
for c in categorical_col:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [ ]:
df.head().T

In [ ]:
df['totalcharges']

In [ ]:
df['totalcharges'] = pd.to_numeric(df.totalcharges, errors = 'coerce')

In [ ]:
df['totalcharges']

In [ ]:
df.totalcharges = df.totalcharges.fillna(0)

In [ ]:
df.totalcharges.isna().sum()

In [ ]:
df.churn.nunique()

In [ ]:
df.churn.head()

In [ ]:
df.churn = (df.churn == 'yes').astype(int)


In [ ]:
df.churn.nunique()

In [ ]:
df.churn.isna().sum()

### Setting up the validation framework

**perform the train/validation/test split with scikit-learn**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
full_train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 1)

In [ ]:
train_df, val_df = train_test_split(full_train_df, test_size = 0.25, random_state = 1)
assert len(df) == len(train_df) + len(test_df) + len(val_df)

In [ ]:
len(train_df), len(val_df), len(test_df)

In [ ]:
train_df.head().T

In [ ]:
train_df = train_df.reset_index(drop = True)
val_df = val_df.reset_index(drop = True)
test_df = test_df.reset_index(drop = True)


In [ ]:
val_df.describe()

**Extract our y variable**

In [ ]:
y_train = train_df.churn.values
y_val = val_df.churn.values
y_test = test_df.churn.values


In [ ]:
del train_df['churn']
del val_df['churn']
del test_df['churn']

In [ ]:
train_df

### EDA [Exploratory Data Analysis]

In [ ]:
full_train_df = full_train_df.reset_index(drop = True)

In [ ]:
full_train_df.isna().sum()

In [ ]:
global_churn_rate = full_train_df.churn.mean()
global_churn_rate 
round(global_churn_rate,3)

In [ ]:
numeric_col = ['tenure', 'monthlycharges', 'totalcharges']

In [ ]:
categorical = ['gender',
 'partner',
 'dependents',
 'phoneservice',
 'multiplelines',
 'internetservice',
 'onlinesecurity',
 'onlinebackup',
 'deviceprotection',
 'techsupport',
 'streamingtv',
 'streamingmovies',
 'contract',
 'paperlessbilling',
 'paymentmethod',
 'seniorcitizen']

In [ ]:
categorical


In [ ]:
full_train_df[categorical].nunique()

### feature importance: Churn rate and risk ratio

Feature importance analysis is part of **(EDA)** identifying which feature affect our target variable .

1. churn rate
2. Risk Ratio
3. Mutual information later


**churn rate**

In [ ]:
full_train_df.head()

In [ ]:
churn_female = full_train_df[full_train_df['gender']=='female'].churn.mean()

In [ ]:
churn_female

In [ ]:
churn_male = full_train_df[full_train_df['gender']=='male'].churn.mean()

churn_male

In [ ]:
global_churn_rate

In [ ]:
# for partner variable

churn_partner = full_train_df[full_train_df['partner'] == 'yes'].churn.mean()
churn_no_partner = full_train_df[full_train_df['partner'] == 'no'].churn.mean()

In [ ]:
churn_partner

In [ ]:
churn_no_partner

In [ ]:
risk_rate_no_partner = churn_no_partner/global_churn_rate
risk_rate_no_partner


In [ ]:
from IPython.display import display

In [ ]:
for c in categorical:
    df_group = full_train_df.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn_rate
    df_group['risk'] = df_group['mean'] / global_churn_rate
    display(df_group)


    

### Feature Importance: Mutual information

Mutual information - concept from information theory, it tells us how much we can learn about one variable if we know the value of another.

In [ ]:
from sklearn.metrics import mutual_info_score

In [ ]:
mutual_info_score(full_train_df.churn, full_train_df.contract)

In [ ]:
mutual_info_score(full_train_df.gender, full_train_df.churn)

In [ ]:
def mutual_info_churn_score(series):

    return mutual_info_score(series, full_train_df.churn)

In [ ]:
mi = full_train_df[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

### Corelation Coeficient

In [ ]:
train_df[numeric_col].dtypes

In [ ]:
full_train_df[numeric_col].corrwith(full_train_df.churn)

In [ ]:
full_train_df[full_train_df.tenure <= 2].churn.mean()

In [ ]:
full_train_df[(full_train_df.tenure > 2) & (full_train_df.tenure <= 12)].churn.mean()

### One-hot encoding

**. Use scikit-learn to encode categorical features**

In [ ]:
train_df.head()

In [ ]:
from sklearn.feature_extraction import DictVectorizer

In [ ]:
dict = train_df[['gender', 'contract']].iloc[:100].to_dict(orient = 'records')


In [ ]:
dv = DictVectorizer(sparse = False)

In [ ]:
dv.fit(dict)

In [ ]:
dv.transform(dict)[0]

In [ ]:
train_dict = train_df[categorical + numeric_col].to_dict(orient = 'records')

In [ ]:
train_dict[0]

In [ ]:
dv = DictVectorizer(sparse = False)

In [ ]:
dv.fit(train_dict)

In [ ]:
X_train = dv.transform(train_dict)

In [ ]:
X_train.shape

In [ ]:
val_dict = val_df[categorical + numeric_col].to_dict(orient = 'records')

In [ ]:
X_val = dv.transform(val_dict)

In [ ]:
X_val.shape

# Logistics Regression

### Training logistics regression with Scikit-Learn

1. Train a model with Scikit-learn
2. Apply it to the validation dataset
3. Calculate the accuracy

In [ ]:
from sklearn.linear_model import LogisticsRegression